In [22]:
from faker import Faker
fake = Faker('en_GB')
Faker.seed(42)

In [23]:
n_sensors = 50
n_readings = 5000

In [24]:
from unidecode import unidecode
import uuid

In [25]:
from datetime import datetime, timedelta

In [26]:
from faker.providers import DynamicProvider

sensor = {}
sensor_type = ['temperature', 'humidity', 'air_quality']

unique_districts = set()
while len(unique_districts) < 8:
    unique_districts.add(fake.administrative_unit())

# 2. Register these 8 districts into a DynamicProvider to override the default method
district_provider = DynamicProvider(
     provider_name="administrative_unit",
     elements=list(unique_districts),
)
fake.add_provider(district_provider)

for i in range(n_sensors):
    latitude = fake.latitude()
    longitude = fake.longitude()
    sensor[i] = {
        'sensor_id': str(uuid.uuid4()),
        'district': fake.administrative_unit(),
        'sensor_type': fake.random_element(elements=sensor_type),
        'latitude_longitude': f"{latitude}, {longitude}",
        'installed_at': fake.date_between(start_date='-3y', end_date='-30d')
    }

print(sensor[0])

{'sensor_id': '25f7fe15-ac58-46fd-a2a5-8ac722e51b07', 'district': 'Gloucestershire', 'sensor_type': 'temperature', 'latitude_longitude': '56.395715, -133.325072', 'installed_at': datetime.date(2023, 8, 16)}


In [27]:
sensor_id = [sensor[i]['sensor_id'] for i in range(n_sensors)]

In [28]:
readings = {}

for i in range(n_readings):
    readings[i] = {
        'reading_id': str(uuid.uuid4()),
        'sensor_id': sensor_id[i % n_sensors],
        'value': round(fake.random.gauss(mu=20, sigma=5), 2) if sensor[i % n_sensors]['sensor_type'] == 'temperature' else (round(fake.random.gauss(mu=50, sigma=10), 2) if sensor[i % n_sensors]['sensor_type'] == 'humidity' else round(fake.random.gauss(mu=100, sigma=20), 2)),
        'unit': '°C' if sensor[i % n_sensors]['sensor_type'] == 'temperature' else ('%' if sensor[i % n_sensors]['sensor_type'] == 'humidity' else 'AQI'),
        'recorded_at': fake.date_time_between(start_date=sensor[i % n_sensors]['installed_at'], end_date=datetime.now()),
        'is_anomaly': fake.boolean(chance_of_getting_true=7)
    }

print(readings[0])

{'reading_id': '2d9ec247-3358-499b-89b6-1a27f522bf50', 'sensor_id': '25f7fe15-ac58-46fd-a2a5-8ac722e51b07', 'value': 35.82, 'unit': '°C', 'recorded_at': datetime.datetime(2025, 3, 30, 4, 54, 8, 333391), 'is_anomaly': False}


In [29]:
import pandas as pd

In [30]:
sensors_df = pd.DataFrame.from_dict(sensor, orient='index')
readings_df = pd.DataFrame.from_dict(readings, orient='index')

In [31]:
sensors_df.to_csv('sensors_raw.csv', index=False)
readings_df.to_csv('readings_raw.csv', index=False)